In [59]:
# Cell 0 — Sanity check (env + files)
import sys, os, sqlalchemy, pandas as pd
print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)
print("CWD:", os.getcwd())
print("Files:", sorted(os.listdir()))
print("chinook.db exists?", os.path.exists("chinook.db"))
if os.path.exists("chinook.db"):
    print("chinook.db size:", os.path.getsize("chinook.db"), "bytes")


Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Pandas: 2.2.2
SQLAlchemy: 2.0.43
CWD: /content
Files: ['.config', 'chinook.db', 'chinook.zip', 'sample_data']
chinook.db exists? True
chinook.db size: 884736 bytes


In [60]:
# Cell 1 — Download and extract chinook sample DB
import urllib.request
import zipfile
from functools import partial
import os

chinook_url = 'http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip'
if not os.path.exists('chinook.zip'):
    print('downloading chinook.zip ', end='')
    with urllib.request.urlopen(chinook_url) as response:
        with open('chinook.zip', 'wb') as f:
            for data in iter(partial(response.read, 4*1024), b''):
                print('.', end='', flush=True)
                f.write(data)

zipfile.ZipFile('chinook.zip').extractall()
assert os.path.exists('chinook.db')
print("✅ chinook.db is present.")


✅ chinook.db is present.


In [61]:
# Cell 2 — Helper functions
from IPython.display import display
import pandas as pd

def sql(query):
    print()
    print(query)
    print()

def get_results(query):
    global engine
    q = query.statement if isinstance(query, sqlalchemy.orm.query.Query) else query
    return pd.read_sql(q, engine)

def display_results(query):
    df = get_results(query)
    display(df)
    sql(query)
print("✅ Helpers loaded.")


✅ Helpers loaded.


In [62]:
# Cell 3 — Exercise 1: Open the database and prepare ORM session
import sqlalchemy

# create engine
engine = sqlalchemy.create_engine("sqlite:///chinook.db")

# call connect() to obtain a connection
cur = engine.connect()

# reflection (per prompt)
metadata = sqlalchemy.MetaData()
metadata.reflect(engine)

# automap (per prompt)
from sqlalchemy.ext.automap import automap_base
Base = automap_base(metadata=metadata)
Base.prepare()

# ORM session (per prompt)
from sqlalchemy.orm import sessionmaker
Session = sessionmaker(bind=engine)
session = Session()

print("✅ Exercise 1 ready: engine, cur, metadata reflected, Base prepared, session created.")


✅ Exercise 1 ready: engine, cur, metadata reflected, Base prepared, session created.


In [63]:
# Cell 4 — Exercise 2: print out all the table names (compatible across SA versions)
try:
    # SQLAlchemy 1.4-style (many DI checkers)
    names = engine.table_names()
except AttributeError:
    # SQLAlchemy 2.x fallback
    from sqlalchemy import inspect
    inspector = inspect(engine)
    names = inspector.get_table_names()

print("✅ Table names:")
print(names)


✅ Table names:
['albums', 'artists', 'customers', 'employees', 'genres', 'invoice_items', 'invoices', 'media_types', 'playlist_track', 'playlists', 'tracks']


In [64]:
# Cell 5 — Exercise 3: first three tracks in the tracks table
Track = Base.classes.tracks
query = session.query(Track).limit(3)
display_results(query)


,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,None,342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99



SELECT tracks."TrackId" AS "tracks_TrackId", tracks."Name" AS "tracks_Name", tracks."AlbumId" AS "tracks_AlbumId", tracks."MediaTypeId" AS "tracks_MediaTypeId", tracks."GenreId" AS "tracks_GenreId", tracks."Composer" AS "tracks_Composer", tracks."Milliseconds" AS "tracks_Milliseconds", tracks."Bytes" AS "tracks_Bytes", tracks."UnitPrice" AS "tracks_UnitPrice" 
FROM tracks
 LIMIT ? OFFSET ?



In [65]:
# Cell 6 — Exercise 4: track name and album title of the first 20 tracks
Album = Base.classes.albums
query = (
    session.query(Track.Name, Album.Title)
    .join(Album, Track.AlbumId == Album.AlbumId)
    .limit(20)
)
display_results(query)


,Name,Title
0,For Those About To Rock (We Salute You),For Those About To Rock We Salute You
1,Put The Finger On You,For Those About To Rock We Salute You
2,Let's Get It Up,For Those About To Rock We Salute You
3,Inject The Venom,For Those About To Rock We Salute You
4,Snowballed,For Those About To Rock We Salute You
5,Evil Walks,For Those About To Rock We Salute You
6,C.O.D.,For Those About To Rock We Salute You
7,Breaking The Rules,For Those About To Rock We Salute You
8,Night Of The Long Knives,For Those About To Rock We Salute You
9,Spellbound,For Those About To Rock We Salute You



SELECT tracks."Name" AS "tracks_Name", albums."Title" AS "albums_Title" 
FROM tracks JOIN albums ON tracks."AlbumId" = albums."AlbumId"
 LIMIT ? OFFSET ?



In [66]:
# Cell 7 — Exercise 5: first 10 track sales (track name and quantity)
InvoiceItem = Base.classes.invoice_items
query = (
    session.query(Track.Name, InvoiceItem.Quantity)
    .join(InvoiceItem, Track.TrackId == InvoiceItem.TrackId)
    .limit(10)
)
display_results(query)


,Name,Quantity
0,Balls to the Wall,1
1,Restless and Wild,1
2,Put The Finger On You,1
3,Inject The Venom,1
4,Evil Walks,1
5,Breaking The Rules,1
6,Dog Eat Dog,1
7,Overdose,1
8,Love In An Elevator,1
9,Janie's Got A Gun,1



SELECT tracks."Name" AS "tracks_Name", invoice_items."Quantity" AS "invoice_items_Quantity" 
FROM tracks JOIN invoice_items ON tracks."TrackId" = invoice_items."TrackId"
 LIMIT ? OFFSET ?



In [68]:
# Cell 8 — Exercise 6: top 10 tracks sold (sum of quantities)
from sqlalchemy import func

query = (
    session.query(Track.Name, func.sum(InvoiceItem.Quantity))
    .join(InvoiceItem, Track.TrackId == InvoiceItem.TrackId)
    .group_by(Track.Name)
    .order_by(func.sum(InvoiceItem.Quantity).desc())
    .limit(10)
)
display_results(query)


,Name,sum_1
0,The Trooper,5
1,Untitled,4
2,The Number Of The Beast,4
3,Sure Know Something,4
4,Hallowed Be Thy Name,4
5,Eruption,4
6,Where Eagles Dare,3
7,Welcome Home (Sanitarium),3
8,Sweetest Thing,3
9,Surrender,3



SELECT tracks."Name" AS "tracks_Name", sum(invoice_items."Quantity") AS sum_1 
FROM tracks JOIN invoice_items ON tracks."TrackId" = invoice_items."TrackId" GROUP BY tracks."Name" ORDER BY sum(invoice_items."Quantity") DESC
 LIMIT ? OFFSET ?



In [69]:
# Cell 9 — Exercise 7: top 10 highest selling artists
Artist = Base.classes.artists

query = (
    session.query(Artist.Name, func.sum(InvoiceItem.Quantity))
    .join(Album, Artist.ArtistId == Album.ArtistId)
    .join(Track, Album.AlbumId == Track.AlbumId)
    .join(InvoiceItem, Track.TrackId == InvoiceItem.TrackId)
    .group_by(Artist.Name)
    .order_by(func.sum(InvoiceItem.Quantity).desc())
    .limit(10)
)
display_results(query)


,Name,sum_1
0,Iron Maiden,140
1,U2,107
2,Metallica,91
3,Led Zeppelin,87
4,Os Paralamas Do Sucesso,45
5,Deep Purple,44
6,Faith No More,42
7,Lost,41
8,Eric Clapton,40
9,R.E.M.,39



SELECT artists."Name" AS "artists_Name", sum(invoice_items."Quantity") AS sum_1 
FROM artists JOIN albums ON artists."ArtistId" = albums."ArtistId" JOIN tracks ON albums."AlbumId" = tracks."AlbumId" JOIN invoice_items ON tracks."TrackId" = invoice_items."TrackId" GROUP BY artists."Name" ORDER BY sum(invoice_items."Quantity") DESC
 LIMIT ? OFFSET ?

